## jul17 — Daily TE reconstruction (final) + monthly-to-daily extrapolation

### Summary
- Official Cboe roll formula (Ra x Rb x Rc): validated, 0.04%p error vs BXM on Oct 21
- Full-year (11 periods) daily TE reconstruction: naive 4.80% -> 3.46% with full fixes
- Remaining gap traced to Trades-only data (no Quotes) -- cannot observe price on
  illiquid days, must approximate
- This notebook: clean final pipeline + Prof. Dodson's monthly-to-daily extrapolation test

In [3]:
# ── Setup ──
import pandas as pd
import numpy as np
import os
import yfinance as yf

DATA_DIR = "../data/raw/cboe_spx_2022/"
RARE_CODES = [136, 13, 118, 137, 40, 41, 42]

spx_year = yf.download("^GSPC", start="2022-01-01", end="2023-01-05")["Close"].squeeze()
spx_year.index = spx_year.index.date

bxm_full = pd.read_csv("../data/raw/BXM_History.csv")
bxm_full["date"] = pd.to_datetime(bxm_full["DATE"], format="%m/%d/%Y").dt.date
bxm_full = bxm_full.sort_values("date").reset_index(drop=True)
bxm_full["bxm_ret"] = bxm_full["BXM"] / bxm_full["BXM"].shift(1) - 1

print(f"SPX trading days loaded: {len(spx_year)}")

[*********************100%***********************]  1 of 1 completed

SPX trading days loaded: 253


In [4]:
# ── Roll date master table ──
roll_dates = pd.DataFrame({
    "roll_date": ["2022-01-21","2022-02-18","2022-03-18","2022-04-14",
                  "2022-05-20","2022-06-17","2022-07-15","2022-08-19",
                  "2022-09-16","2022-10-21","2022-11-18","2022-12-16"],
    "SOQ": [4472.07, 4383.70, 4409.35, 4452.07, 3937.64, 3663.76,
            3839.81, 4258.21, 3871.24, 3656.28, 3983.42, 3871.47],
    "K_new": [4475, 4365, 4415, 4430, 3885, 3655, 3850, 4235, 3855, 3680, 3950, 3850],
    "C_VWAP_new": [89.06, 104.36, 99.99, 92.51, 97.64, 122.02,
                   107.05, 85.29, 118.53, 135.64, 96.31, 97.48],
})
roll_dates["roll_date"] = pd.to_datetime(roll_dates["roll_date"]).dt.date
roll_dates["K_old"] = roll_dates["K_new"].shift(1)
roll_dates["C_settle"] = np.maximum(0, roll_dates["SOQ"] - roll_dates["K_old"])
roll_dates["expiration"] = roll_dates["roll_date"].shift(-1)
roll_dates.loc[roll_dates["roll_date"] == pd.Timestamp("2022-12-16").date(), "expiration"] = pd.Timestamp("2023-01-20").date()

print(roll_dates)

     roll_date      SOQ  K_new  C_VWAP_new   K_old  C_settle  expiration
0   2022-01-21  4472.07   4475       89.06     NaN       NaN  2022-02-18
1   2022-02-18  4383.70   4365      104.36  4475.0      0.00  2022-03-18
2   2022-03-18  4409.35   4415       99.99  4365.0     44.35  2022-04-14
3   2022-04-14  4452.07   4430       92.51  4415.0     37.07  2022-05-20
4   2022-05-20  3937.64   3885       97.64  4430.0      0.00  2022-06-17
5   2022-06-17  3663.76   3655      122.02  3885.0      0.00  2022-07-15
6   2022-07-15  3839.81   3850      107.05  3655.0    184.81  2022-08-19
7   2022-08-19  4258.21   4235       85.29  3850.0    408.21  2022-09-16
8   2022-09-16  3871.24   3855      118.53  4235.0      0.00  2022-10-21
9   2022-10-21  3656.28   3680      135.64  3855.0      0.00  2022-11-18
10  2022-11-18  3983.42   3950       96.31  3680.0    303.42  2022-12-16
11  2022-12-16  3871.47   3850       97.48  3950.0      0.00  2023-01-20


In [5]:
# ── Final functions: trade_condition_id filter + delta-projected carry-forward ──
def get_option_daily_series(strike, expiration, start_date, end_date):
    valid_days = [d for d in spx_year.index if start_date.date() <= d <= end_date.date()]
    rows = []
    last_known_C, last_known_delta, last_known_S = None, None, None

    for d in valid_days:
        fname = f"UnderlyingOptionsTradesCalcs_{d.strftime('%Y-%m-%d')}.csv"
        fpath = os.path.join(DATA_DIR, fname)
        S_today = spx_year.loc[d]

        def carry_forward():
            if last_known_C is None:
                return None
            return max(0, last_known_C + last_known_delta * (S_today - last_known_S))

        if not os.path.exists(fpath):
            C_approx = carry_forward()
            if C_approx is not None:
                rows.append({"date": d, "C_t": C_approx, "n_quotes": 0, "approx": True})
            continue

        df = pd.read_csv(fpath)
        df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])
        mask = (
            (df["root"] == "SPX") & (df["option_type"] == "C")
            & (df["strike"] == strike)
            & (df["expiration"] == expiration.strftime("%Y-%m-%d"))
            & (~df["trade_condition_id"].isin(RARE_CODES))
        )
        day_df = df[mask].sort_values("quote_datetime")

        if day_df.empty:
            C_approx = carry_forward()
            if C_approx is not None:
                rows.append({"date": d, "C_t": C_approx, "n_quotes": 0, "approx": True})
            continue

        before_4pm = day_df[day_df["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
        if before_4pm.empty:
            before_4pm = day_df
        last_q = before_4pm.iloc[-1]
        last_known_C = (last_q["best_bid"] + last_q["best_ask"]) / 2
        last_known_delta = last_q["trade_delta"]
        last_known_S = S_today
        rows.append({"date": d, "C_t": last_known_C, "n_quotes": len(before_4pm), "approx": False})

    return pd.DataFrame(rows)


def compute_roll_leg(roll_date, K_old, K_new, K_new_expiration, SOQ, C_settle, C_VWAP_new, S_prev, C_prev):
    fname = f"UnderlyingOptionsTradesCalcs_{roll_date.strftime('%Y-%m-%d')}.csv"
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath)
    df["quote_datetime"] = pd.to_datetime(df["quote_datetime"])

    new_mask = (
        (df["root"] == "SPX") & (df["option_type"] == "C")
        & (df["strike"] == K_new)
        & (df["expiration"] == K_new_expiration.strftime("%Y-%m-%d"))
        & (~df["trade_condition_id"].isin(RARE_CODES))
    )
    new_day = df[new_mask].sort_values("quote_datetime")

    window_mask = (
        (new_day["quote_datetime"].dt.time >= pd.Timestamp("11:30:00").time())
        & (new_day["quote_datetime"].dt.time <= pd.Timestamp("13:30:00").time())
    )
    window_trades = new_day[window_mask]
    S_VWAV = (window_trades["underlying_bid"] * window_trades["trade_size"]).sum() / window_trades["trade_size"].sum()

    before_4pm = new_day[new_day["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
    C_t_new = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2

    S_t = spx_year.loc[roll_date]

    Ra = (SOQ + 0 - C_settle) / (S_prev - C_prev) - 1
    Rb = S_VWAV / SOQ - 1
    Rc = (S_t - C_t_new) / (S_VWAV - C_VWAP_new) - 1
    roll_ret = (1 + Ra) * (1 + Rb) * (1 + Rc) - 1

    return roll_ret, S_t, C_t_new

print("Functions loaded.")

Functions loaded.


In [6]:
# ── Baseline: Jan 21 close ──
jan21_file = os.path.join(DATA_DIR, "UnderlyingOptionsTradesCalcs_2022-01-21.csv")
jan21_df = pd.read_csv(jan21_file)
jan21_df["quote_datetime"] = pd.to_datetime(jan21_df["quote_datetime"])

mask = (
    (jan21_df["root"] == "SPX") & (jan21_df["option_type"] == "C")
    & (jan21_df["strike"] == 4475)
    & (jan21_df["expiration"] == "2022-02-18")
    & (~jan21_df["trade_condition_id"].isin(RARE_CODES))
)
jan21_opt = jan21_df[mask].sort_values("quote_datetime")
before_4pm = jan21_opt[jan21_opt["quote_datetime"].dt.time < pd.Timestamp("16:00:00").time()]
C_prev = (before_4pm.iloc[-1]["best_bid"] + before_4pm.iloc[-1]["best_ask"]) / 2
S_prev = spx_year.loc[pd.Timestamp("2022-01-21").date()]

print(f"S_prev={S_prev}, C_prev={C_prev}")
assert abs(S_prev - 4397.94) < 1 and abs(C_prev - 80.45) < 1, "Baseline reset failed!"
print("Baseline OK.")

S_prev=4397.93994140625, C_prev=80.44999999999999
Baseline OK.


In [7]:
# ── Main loop: full-year daily reconstruction ──
all_returns = {}

for i in range(len(roll_dates) - 1):
    period_start = roll_dates.iloc[i]["roll_date"]
    period_end = roll_dates.iloc[i + 1]["roll_date"]
    held_strike = roll_dates.iloc[i]["K_new"]
    held_exp = roll_dates.iloc[i]["expiration"]

    non_roll_days = pd.bdate_range(period_start, period_end)[1:-1]
    opt_series = get_option_daily_series(held_strike, held_exp, non_roll_days.min(), non_roll_days.max())

    for _, row in opt_series.sort_values("date").iterrows():
        d = row["date"]
        S_t = spx_year.loc[d]
        C_t = row["C_t"]
        ret = (S_t - C_t) / (S_prev - C_prev) - 1
        all_returns[d] = ret
        S_prev, C_prev = S_t, C_t

    row_next = roll_dates.iloc[i + 1]
    roll_ret, S_t, C_t = compute_roll_leg(
        roll_date=period_end,
        K_old=row_next["K_old"], K_new=row_next["K_new"],
        K_new_expiration=row_next["expiration"],
        SOQ=row_next["SOQ"], C_settle=row_next["C_settle"],
        C_VWAP_new=row_next["C_VWAP_new"],
        S_prev=S_prev, C_prev=C_prev,
    )
    all_returns[period_end] = roll_ret
    S_prev, C_prev = S_t, C_t

our_returns = pd.Series(all_returns).sort_index()
print(f"Total days reconstructed: {len(our_returns)}")

Total days reconstructed: 228


In [8]:
# ── Compare against official BXM ──
our_df = our_returns.rename("our_ret").to_frame()
our_df.index.name = "date"

final_check = our_df.merge(bxm_full.set_index("date")[["bxm_ret"]], left_index=True, right_index=True, how="inner")
final_check["te"] = final_check["our_ret"] - final_check["bxm_ret"]

daily_te = final_check["te"].std() * (252**0.5)
our_vol = final_check["our_ret"].std() * (252**0.5)
bxm_vol = final_check["bxm_ret"].std() * (252**0.5)

print(f"N days matched: {len(final_check)}")
print(f"Our ann. vol:  {our_vol:.4%}")
print(f"BXM ann. vol:  {bxm_vol:.4%}")
print(f"Full-year daily TE: {daily_te:.4%}")
print(f"Mean daily TE:      {final_check['te'].mean():.4%}")

N days matched: 228
Our ann. vol:  17.2780%
BXM ann. vol:  16.6190%
Full-year daily TE: 3.4671%
Mean daily TE:      -0.0054%


In [10]:
# ── Extrapolation test: does the monthly-to-daily TE gap match known frictions? ──
monthly_te = 0.0093       # expiry-to-expiry annualized TE (Stage 1)
vwap_slippage = 0.0141    # annualized VWAP slippage component (Stage 2, a MEAN drag, not a std)
naive_daily_te = 0.0480   # naive daily TE (original baseline)
our_daily_te = daily_te   # this notebook's carefully reconstructed daily TE

# direct mechanistic evidence from this notebook: roll-day reconstruction accuracy
roll_day_error_pp = 0.04  # percentage points, Oct 21 roll vs official BXM (from earlier test)

gap_naive = naive_daily_te - monthly_te
gap_reconstructed = our_daily_te - monthly_te
vwap_share_naive = vwap_slippage / gap_naive
vwap_share_reconstructed = vwap_slippage / gap_reconstructed

print(f"Monthly (expiry-to-expiry) TE:        {monthly_te:.4%}")
print(f"Naive daily TE:                        {naive_daily_te:.4%}")
print(f"Carefully reconstructed daily TE:      {our_daily_te:.4%}")
print(f"\nGap: naive daily vs monthly:          {gap_naive:.4%}")
print(f"Gap: reconstructed daily vs monthly:   {gap_reconstructed:.4%}")
print(f"\nVWAP slippage (monthly, Stage 2):      {vwap_slippage:.4%}")
print(f"VWAP slippage as share of naive gap:        {vwap_share_naive:.1%}")
print(f"VWAP slippage as share of reconstructed gap: {vwap_share_reconstructed:.1%}")

print(f"\nDirect mechanistic check: roll-day (Ra x Rb x Rc) reconstruction error vs "
      f"official BXM = {roll_day_error_pp:.2f}pp on the one roll date tested.")

print("""
Interpretation:
- VWAP slippage (a MEAN drag) and daily TE (a STD of noise) are different kinds of
  statistics -- the ratios above are a rough order-of-magnitude check, not a causal
  variance decomposition.
- Direct mechanistic evidence outweighs this ratio: reconstructing the roll date itself
  with the official 3-leg formula, where VWAP actually enters the calculation, matched
  official BXM to within ~0.04pp. Since VWAP is applied correctly there and the error
  is tiny, VWAP is not the source of the remaining daily noise.
- The remaining gap is better explained by illiquid-day quote unavailability (Trades-only
  data), which this notebook approximates rather than observes, plus other frictions
  Prof. Dodson flagged as out of scope (creation/redemption flows, fees).
""")

Monthly (expiry-to-expiry) TE:        0.9300%
Naive daily TE:                        4.8000%
Carefully reconstructed daily TE:      3.4671%

Gap: naive daily vs monthly:          3.8700%
Gap: reconstructed daily vs monthly:   2.5371%

VWAP slippage (monthly, Stage 2):      1.4100%
VWAP slippage as share of naive gap:        36.4%
VWAP slippage as share of reconstructed gap: 55.6%

Direct mechanistic check: roll-day (Ra x Rb x Rc) reconstruction error vs official BXM = 0.04pp on the one roll date tested.

Interpretation:
- VWAP slippage (a MEAN drag) and daily TE (a STD of noise) are different kinds of
  statistics -- the ratios above are a rough order-of-magnitude check, not a causal
  variance decomposition.
- Direct mechanistic evidence outweighs this ratio: reconstructing the roll date itself
  with the official 3-leg formula, where VWAP actually enters the calculation, matched
  official BXM to within ~0.04pp. Since VWAP is applied correctly there and the error
  is tiny, VWAP is n